In [ ]:
import torch
import pandas as pd
import os

from tabicl import TabICLClassifier

In [ ]:
train_df = pd.read_csv("")
test_df1 = pd.read_csv("")
test_df2 = pd.read_csv("")

In [ ]:
print(train_df.shape)
print(test_df1.shape)
print(test_df2.shape)

In [ ]:
train_df['Revised_final_diagnosis'].value_counts()

In [ ]:
test_df1['Revised_final_diagnosis'].value_counts()

In [ ]:
test_df2['Revised_final_diagnosis'].value_counts()

In [ ]:
disease_classes_to_test = ['Acute lymphoblastic leukemia', 'Chronic myeloid leukemia', 'Acute myeloid leukemia non M3', 'Multiple myeloma', 'Aplastic anemia', 'Chronic lymphocytic leukemia', 'Acute promyelocytic leukemia', 'Acute leukemia', 'Eosinophilia', 'Primary myelofibrosis']

train_set_to_be_used = train_df[train_df['Revised_final_diagnosis'].isin(disease_classes_to_test)]
test_set_to_be_used1 = test_df1[test_df1['Revised_final_diagnosis'].isin(disease_classes_to_test)]
test_set_to_be_used2 = test_df2[test_df2['Revised_final_diagnosis'].isin(disease_classes_to_test)]

In [ ]:
train_set_to_be_used['Revised_final_diagnosis'].value_counts()

In [ ]:
test_set_to_be_used1['Revised_final_diagnosis'].value_counts()

In [ ]:
test_set_to_be_used2['Revised_final_diagnosis'].value_counts()

In [ ]:
id_columns = ['pseudo_patient_id', 'Analyzer_model', 'Analyzer_id']
feature_columns = ['WBC(10^3/uL)', 'RBC(10^6/uL)', 'HGB(g/dL)', 'HCT(%)', 'MCV(fL)', 'MCH(pg)', 'MCHC(g/dL)', 'PLT(10^3/uL)', 'NRBC#(10^3/uL)', 'NRBC%(%)', 'NEUT#(10^3/uL)', 'LYMPH#(10^3/uL)', 'MONO#(10^3/uL)', 'EO#(10^3/uL)', 'BASO#(10^3/uL)', 'NEUT%(%)', 'LYMPH%(%)', 'MONO%(%)', 'EO%(%)', 'BASO%(%)', 'IG#(10^3/uL)', 'IG%(%)', '[PLT-I(10^3/uL)]', 'MicroR(%)', 'MacroR(%)', '[TNC(10^3/uL)]', '[WBC-N(10^3/uL)]', '[TNC-N(10^3/uL)]', '[BA-N#(10^3/uL)]', '[BA-N%(%)]', '[WBC-D(10^3/uL)]', '[TNC-D(10^3/uL)]', '[NEUT#&(10^3/uL)]', '[NEUT%&(%)]', '[LYMP#&(10^3/uL)]', '[LYMP%&(%)]', '[HFLC#(10^3/uL)]', '[HFLC%(%)]', '[BA-D#(10^3/uL)]', '[BA-D%(%)]', '[NE-SSC(ch)]', '[NE-SFL(ch)]', '[NE-FSC(ch)]', '[LY-X(ch)]', '[LY-Y(ch)]', '[LY-Z(ch)]', '[MO-X(ch)]', '[MO-Y(ch)]', '[MO-Z(ch)]', '[NE-WX]', '[NE-WY]', '[NE-WZ]', '[LY-WX]', '[LY-WY]', '[LY-WZ]', '[MO-WX]', '[MO-WY]', '[MO-WZ]']
missing_val_feature_columns = ['RDW-SD(fL)', 'RDW-CV(%)', 'PDW(fL)', 'MPV(fL)', 'P-LCR(%)', 'PCT(%)']
label_columns = ['Revised_final_diagnosis']

In [ ]:
X_train = train_set_to_be_used[feature_columns].values
y_train = train_set_to_be_used[label_columns].values
X_test1 = test_set_to_be_used1[feature_columns].values
y_test1 = test_set_to_be_used1[label_columns].values
X_test2 = test_set_to_be_used2[feature_columns].values
y_test2 = test_set_to_be_used2[label_columns].values

In [ ]:
print(type(X_train), X_train.shape)
print(type(y_train), y_train.shape)
print(type(X_test1), X_test1.shape)
print(type(y_test1), y_test1.shape)
print(type(X_test2), X_test2.shape)
print(type(y_test2), y_test2.shape)

In [ ]:
# find missing values in these numpy arrays
print("Missing values in X_train:", pd.isnull(X_train).sum())
print("Missing values in y_train:", pd.isnull(y_train).sum())
print("Missing values in X_test:", pd.isnull(X_test1).sum())
print("Missing values in y_test:", pd.isnull(y_test1).sum())
print("Missing values in X_test:", pd.isnull(X_test2).sum())
print("Missing values in y_test:", pd.isnull(y_test2).sum())

In [ ]:
print(y_train)

In [ ]:
clf = TabICLClassifier()
clf.fit(X_train, y_train)

In [ ]:
labels1 = clf.predict(X_test1)
labels2 = clf.predict(X_test2)

In [ ]:
print(clf.classes_)

In [ ]:
print(labels1.shape)
print(labels2.shape)

In [ ]:
print(labels1)

In [ ]:
# add the predicted labels to the test_set_to_be_used dataframe

test_set_to_be_used1['Predicted_diagnosis'] = labels1
test_set_to_be_used2['Predicted_diagnosis'] = labels2

In [ ]:
test_set_to_be_used1.drop(columns=missing_val_feature_columns, inplace=True)
test_set_to_be_used2.drop(columns=missing_val_feature_columns, inplace=True)

In [ ]:
test_set_to_be_used1.head()

In [ ]:
test_set_to_be_used2.head()

In [ ]:
print(test_set_to_be_used1.shape)
print(test_set_to_be_used2.shape)

In [ ]:
test_set_to_be_used1.to_csv("prospective_predicted_results.csv", index=False)
test_set_to_be_used2.to_csv("crossinstrumental_predicted_results.csv", index=False)

In [ ]:
# calculate classification metrics on the test set Revised_final_diagnosis and Predicted_diagnosis columns
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
print(classification_report(test_set_to_be_used1['Revised_final_diagnosis'], test_set_to_be_used1['Predicted_diagnosis']))

print("Confusion Matrix:")
print(confusion_matrix(test_set_to_be_used1['Revised_final_diagnosis'], test_set_to_be_used1['Predicted_diagnosis']))

In [ ]:
print("Classification Report:")
print(classification_report(test_set_to_be_used2['Revised_final_diagnosis'], test_set_to_be_used2['Predicted_diagnosis']))

print("Confusion Matrix:")
print(confusion_matrix(test_set_to_be_used2['Revised_final_diagnosis'], test_set_to_be_used2['Predicted_diagnosis']))